# Transition Into High Activity and Heavy Imbalance

This notebook studies how the market enters the regime where total market-order activity is high and order book imbalance is heavy.

The target regime is:

$$
\lambda_{total}(t; W) \in \text{high bucket} \quad \text{and} \quad \text{imbalance family}_t = \text{heavy}
$$

Plain-language goal: identify which states tend to precede this regime, how persistent it is, and what the book and price look like around entry.


## 1. Parameters

These parameters intentionally mirror the first imbalance notebook so results are comparable.


In [ ]:
# Notebook parameters.

SYMBOL = "BTCUSDC"
DAYS = ["20260225"]

BOOK_REPLAY_LEVELS = 5
IMBALANCE_LEVELS = 1
N_REGIMES = 5

# Total market-order intensity window, in milliseconds.
LAMBDA_WINDOW_MS = 3000
LAMBDA_BUCKET_LABELS = ["low", "mid", "high"]

TARGET_ACTIVITY_BUCKET = "high"
TARGET_IMBALANCE_FAMILY = "heavy"
TARGET_DIRECTIONAL_LABELS = ["strong buy-heavy", "strong sell-heavy"]

PRE_ENTRY_STEPS = [1, 2, 5, 10]
EVENT_STUDY_OFFSETS_MS = [-3000, -2000, -1000, -500, -250, -100, 0, 100, 250, 500, 1000, 2000, 3000]
MAX_EVENT_STUDY_ENTRIES = 5000

SHOW_REPLAY_PROGRESS = True
REPLAY_ON_GAP = "skip-segment"


## 2. Setup and Data Loading

This notebook is self-contained. It loads the same cached book-level and trade tables used by the first imbalance notebook.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_backtester_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd()]
    for candidate in candidates:
        if (candidate / "stats").is_dir() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the exchange-data-backtester project root")


def resolve_day_dir(project_root: Path, *, symbol: str, day: str) -> Path:
    candidates = [
        project_root.parent / "exchange-data-recorder" / "data" / "binance" / symbol / day,
        project_root.parent / "exchange-data-recorder" / "data" / symbol / day,
        project_root.parent / "ExchangeDataRecorder" / "data" / "binance" / symbol / day,
        project_root.parent / "ExchangeDataRecorder" / "data" / symbol / day,
        project_root / "data" / "binance" / symbol / day,
        project_root / "data" / symbol / day,
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "schema.json").is_file():
            return candidate
    raise FileNotFoundError(f"Could not locate a recorder day folder for {symbol}/{day}")


PROJECT_ROOT = find_backtester_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats.features.book import compute_depth_imbalance, compute_mid_spread
from stats.io import load_day
from stats.notebook import replay_summary
from stats.tables import get_or_build_book_levels_table, get_or_build_trades_table

loaded_days = []
book_parts = []
trade_parts = []

for day in DAYS:
    day_dir = resolve_day_dir(PROJECT_ROOT, symbol=SYMBOL, day=day)
    dataset = load_day(day_dir)
    replay_info = replay_summary(dataset, replay_on_gap=REPLAY_ON_GAP)

    if IMBALANCE_LEVELS > BOOK_REPLAY_LEVELS:
        raise ValueError("IMBALANCE_LEVELS cannot exceed BOOK_REPLAY_LEVELS")

    print(f"Loading {SYMBOL} {day}: book levels top_n={BOOK_REPLAY_LEVELS}, replay_on_gap={REPLAY_ON_GAP}", flush=True)
    book_day = get_or_build_book_levels_table(
        dataset,
        top_n=BOOK_REPLAY_LEVELS,
        on_gap=REPLAY_ON_GAP,
        show_progress=SHOW_REPLAY_PROGRESS,
    )
    print(f"Loaded {SYMBOL} {day}: {len(book_day):,} book rows", flush=True)

    print(f"Loading {SYMBOL} {day}: trades", flush=True)
    trades_day = get_or_build_trades_table(dataset)
    print(f"Loaded {SYMBOL} {day}: {len(trades_day):,} trade rows", flush=True)

    book_day = book_day.copy()
    trades_day = trades_day.copy()
    book_day["source_day"] = day
    trades_day["source_day"] = day

    book_parts.append(book_day)
    trade_parts.append(trades_day)
    loaded_days.append({"day": day, "day_dir": str(day_dir), "book_rows": len(book_day), "trade_rows": len(trades_day), **replay_info})

book_levels = pd.concat(book_parts, ignore_index=True).sort_values(["source_day", "recv_time_ms", "recv_seq"])
trades = pd.concat(trade_parts, ignore_index=True).sort_values(["source_day", "recv_time_ms", "recv_seq"])

load_summary = pd.DataFrame(loaded_days)
display(load_summary)


## 3. Feature Construction

We construct the same base features as in `01_imbalance_basics.ipynb`:

- midprice and spread
- depth imbalance `rho`
- discrete imbalance state `Z`
- total market-order intensity over `LAMBDA_WINDOW_MS`
- joint state combining activity bucket and imbalance family


In [ ]:
def regime_labels(n_regimes: int) -> dict[int, str]:
    if n_regimes == 5:
        return {
            1: "strong sell-heavy",
            2: "mild sell-heavy",
            3: "neutral",
            4: "mild buy-heavy",
            5: "strong buy-heavy",
        }
    return {idx: f"Z={idx}" for idx in range(1, n_regimes + 1)}


def add_imbalance_features(book: pd.DataFrame, *, depth_levels: int, n_regimes: int) -> pd.DataFrame:
    out = book.copy()
    mid_spread = compute_mid_spread(out)
    depth_imbalance = compute_depth_imbalance(out, levels=[depth_levels])

    out["mid"] = mid_spread["mid"]
    out["spread"] = mid_spread["spread"]
    out["spread_bps"] = mid_spread["spread_bps"]
    out["rho"] = depth_imbalance[f"imbalance_{depth_levels}"].clip(-1, 1)

    edges = np.linspace(-1.0, 1.0, n_regimes + 1)
    labels = list(range(1, n_regimes + 1))
    out["Z"] = pd.cut(out["rho"], bins=edges, labels=labels, include_lowest=True).astype("Int64")
    out["Z_label"] = out["Z"].map(regime_labels(n_regimes)).astype("string")

    out = out.sort_values(["source_day", "epoch_id", "segment_index", "recv_time_ms", "recv_seq"]).reset_index(drop=True)
    group_cols = ["source_day", "epoch_id", "segment_index"]
    next_time = out.groupby(group_cols, dropna=False)["recv_time_ms"].shift(-1)
    out["dt_ms"] = (next_time - out["recv_time_ms"]).clip(lower=0).fillna(0)
    return out


def _count_in_window(times: np.ndarray, row_times: np.ndarray, window_ms: int) -> np.ndarray:
    right = np.searchsorted(times, row_times, side="right")
    left = np.searchsorted(times, row_times - window_ms, side="right")
    return right - left


def add_total_intensity(features: pd.DataFrame, trades: pd.DataFrame, window_ms: int) -> pd.DataFrame:
    out = features.copy()
    out[f"lambda_total_raw_{window_ms}ms"] = 0
    out[f"lambda_total_rate_{window_ms}ms"] = 0.0

    for day, idx in out.groupby("source_day", dropna=False).groups.items():
        day_trades = trades[trades["source_day"] == day]
        row_times = out.loc[idx, "recv_time_ms"].to_numpy(dtype="float64")
        trade_times = day_trades["recv_time_ms"].to_numpy(dtype="float64")
        counts = _count_in_window(trade_times, row_times, window_ms)
        out.loc[idx, f"lambda_total_raw_{window_ms}ms"] = counts
        out.loc[idx, f"lambda_total_rate_{window_ms}ms"] = counts / window_ms
    return out


def add_activity_bucket(frame: pd.DataFrame, source_col: str, bucket_col: str, *, labels: list[str]) -> pd.DataFrame:
    out = frame.copy()
    out[bucket_col] = pd.Series(pd.NA, index=out.index, dtype="string")
    valid = out[source_col].notna()
    if not valid.any():
        return out

    ranks = out.loc[valid, source_col].rank(method="first")
    out.loc[valid, bucket_col] = pd.cut(ranks, bins=len(labels), labels=labels, include_lowest=True).astype("string")
    return out


def add_imbalance_family(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    z_label = out["Z_label"].astype("string")
    out["imbalance_family"] = np.select(
        [
            z_label.str.contains("strong buy-heavy", na=False) | z_label.str.contains("strong sell-heavy", na=False),
            z_label.str.contains("mild buy-heavy", na=False) | z_label.str.contains("mild sell-heavy", na=False),
            z_label.str.contains("neutral", na=False),
        ],
        ["heavy", "mid", "neutral"],
        default=pd.NA,
    )
    out["imbalance_family"] = out["imbalance_family"].astype("string")
    return out


features = add_imbalance_features(book_levels, depth_levels=IMBALANCE_LEVELS, n_regimes=N_REGIMES)
features = add_total_intensity(features, trades, LAMBDA_WINDOW_MS)
features = add_activity_bucket(features, f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "lambda_total_bucket", labels=LAMBDA_BUCKET_LABELS)
features = add_imbalance_family(features)
features["joint_state"] = features["lambda_total_bucket"].astype("string") + "|" + features["imbalance_family"].astype("string")
features["is_high_heavy"] = features["lambda_total_bucket"].eq(TARGET_ACTIVITY_BUCKET) & features["imbalance_family"].eq(TARGET_IMBALANCE_FAMILY)
features["is_high_strong_buy"] = features["lambda_total_bucket"].eq(TARGET_ACTIVITY_BUCKET) & features["Z_label"].eq("strong buy-heavy")
features["is_high_strong_sell"] = features["lambda_total_bucket"].eq(TARGET_ACTIVITY_BUCKET) & features["Z_label"].eq("strong sell-heavy")
features["ts"] = pd.to_datetime(features["recv_time_ms"], unit="ms", utc=True)

summary_cols = ["source_day", "recv_time_ms", "mid", "rho", "Z", "Z_label", f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "lambda_total_bucket", "imbalance_family", "joint_state", "is_high_heavy"]
display(features[summary_cols].head())
display(features[["rho", f"lambda_total_raw_{LAMBDA_WINDOW_MS}ms", f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "dt_ms"]].describe())


## 4. Joint State Occupancy

Before studying transitions, first measure how much time and how many observations fall into each joint state.

The joint state is:

$$
S_t = (\lambda\_bucket_t, \text{imbalance family}_t)
$$


In [ ]:
joint_occupancy = (
    features.dropna(subset=["joint_state"])
    .groupby(["lambda_total_bucket", "imbalance_family", "joint_state"], observed=True)
    .agg(
        observations=("joint_state", "size"),
        total_time_ms=("dt_ms", "sum"),
        mean_rho=("rho", "mean"),
        mean_lambda_rate=(f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "mean"),
    )
    .reset_index()
)
joint_occupancy["time_share"] = joint_occupancy["total_time_ms"] / joint_occupancy["total_time_ms"].sum()
joint_occupancy = joint_occupancy.sort_values(["lambda_total_bucket", "imbalance_family"])

display(joint_occupancy)

directional_target_occupancy = (
    features[features["lambda_total_bucket"].eq(TARGET_ACTIVITY_BUCKET) & features["Z_label"].isin(TARGET_DIRECTIONAL_LABELS)]
    .groupby(["Z", "Z_label"], observed=True)
    .agg(observations=("Z", "size"), total_time_ms=("dt_ms", "sum"), mean_rho=("rho", "mean"), mean_lambda_rate=(f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "mean"))
    .reset_index()
    .sort_values("Z")
)
display(directional_target_occupancy)


## 5. Joint State Transition Matrix

This matrix estimates:

$$
P(S_{t+1}=j \mid S_t=i)
$$

It tells us which regimes are likely to lead into `high|heavy`, and whether `high|heavy` tends to persist.


In [ ]:
def build_joint_transitions(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["source_day", "epoch_id", "segment_index"]
    for keys, part in frame.dropna(subset=["joint_state"]).groupby(group_cols, dropna=False):
        part = part.sort_values(["recv_time_ms", "recv_seq"]).copy()
        nxt = part.shift(-1)
        valid = nxt["joint_state"].notna()
        if valid.any():
            rows.append(
                pd.DataFrame(
                    {
                        "source_day": part.loc[valid, "source_day"].to_numpy(),
                        "from_joint_state": part.loc[valid, "joint_state"].astype(str).to_numpy(),
                        "to_joint_state": nxt.loc[valid, "joint_state"].astype(str).to_numpy(),
                        "from_Z_label": part.loc[valid, "Z_label"].astype(str).to_numpy(),
                        "to_Z_label": nxt.loc[valid, "Z_label"].astype(str).to_numpy(),
                        "from_is_high_heavy": part.loc[valid, "is_high_heavy"].to_numpy(dtype=bool),
                        "to_is_high_heavy": nxt.loc[valid, "is_high_heavy"].to_numpy(dtype=bool),
                        "dt_ms": part.loc[valid, "dt_ms"].to_numpy(dtype="float64"),
                    }
                )
            )
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


joint_transitions = build_joint_transitions(features)
state_order = [f"{bucket}|{family}" for bucket in LAMBDA_BUCKET_LABELS for family in ["heavy", "mid", "neutral"]]

transition_counts = pd.crosstab(joint_transitions["from_joint_state"], joint_transitions["to_joint_state"])
transition_counts = transition_counts.reindex(index=state_order, columns=state_order, fill_value=0)
transition_probs = transition_counts.div(transition_counts.sum(axis=1).replace(0, np.nan), axis=0)

display(transition_counts)
display(transition_probs.style.format("{:.3f}").set_caption("Joint-state transition probabilities by book update count"))

incoming_high_heavy = (
    joint_transitions[joint_transitions["to_joint_state"].eq(f"{TARGET_ACTIVITY_BUCKET}|{TARGET_IMBALANCE_FAMILY}")]
    .groupby(["from_joint_state", "to_joint_state"], observed=True)
    .agg(transitions=("to_joint_state", "size"), mean_dt_ms=("dt_ms", "mean"), total_dt_ms=("dt_ms", "sum"))
    .reset_index()
    .sort_values("transitions", ascending=False)
)
display(incoming_high_heavy)


## 6. Entries Into High Activity and Heavy Imbalance

A transition into the target state is an entry when:

$$
\text{is\_high\_heavy}_t = 1 \quad \text{and} \quad \text{is\_high\_heavy}_{t-1} = 0
$$

This avoids counting every row inside a persistent high/heavy run as a new transition.


In [ ]:
def mark_target_entries(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.sort_values(["source_day", "epoch_id", "segment_index", "recv_time_ms", "recv_seq"]).copy()
    group_cols = ["source_day", "epoch_id", "segment_index"]
    prev_target = out.groupby(group_cols, dropna=False)["is_high_heavy"].shift(1).fillna(False)
    out["is_high_heavy_entry"] = out["is_high_heavy"] & ~prev_target
    for step in PRE_ENTRY_STEPS:
        out[f"prev_{step}_joint_state"] = out.groupby(group_cols, dropna=False)["joint_state"].shift(step)
        out[f"prev_{step}_Z_label"] = out.groupby(group_cols, dropna=False)["Z_label"].shift(step)
        out[f"prev_{step}_lambda_bucket"] = out.groupby(group_cols, dropna=False)["lambda_total_bucket"].shift(step)
    return out


features_with_entries = mark_target_entries(features)
entry_cols = [
    "source_day",
    "epoch_id",
    "segment_index",
    "recv_time_ms",
    "ts",
    "mid",
    "rho",
    "Z",
    "Z_label",
    f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms",
    "lambda_total_bucket",
    "imbalance_family",
    "joint_state",
] + [f"prev_{step}_joint_state" for step in PRE_ENTRY_STEPS] + [f"prev_{step}_Z_label" for step in PRE_ENTRY_STEPS]
entries = features_with_entries.loc[features_with_entries["is_high_heavy_entry"], entry_cols].copy()
entries["entry_direction"] = entries["Z_label"].where(entries["Z_label"].isin(TARGET_DIRECTIONAL_LABELS), "other-heavy")

display(entries.head())
display(entries["entry_direction"].value_counts(dropna=False).rename_axis("entry_direction").reset_index(name="entries"))

precursor_rows = []
for step in PRE_ENTRY_STEPS:
    table = (
        entries.groupby([f"prev_{step}_joint_state", "entry_direction"], dropna=False, observed=True)
        .size()
        .reset_index(name="entries")
        .rename(columns={f"prev_{step}_joint_state": "previous_joint_state"})
    )
    table["steps_before_entry"] = step
    precursor_rows.append(table)
precursor_summary = pd.concat(precursor_rows, ignore_index=True) if precursor_rows else pd.DataFrame()
precursor_summary = precursor_summary.sort_values(["steps_before_entry", "entries"], ascending=[True, False])
display(precursor_summary)


## 7. Duration of High/Heavy Episodes

Entries are point events. Episodes measure how long the target regime persists after entry.


In [ ]:
def build_target_episodes(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["source_day", "epoch_id", "segment_index"]
    for _, part in frame.sort_values(["source_day", "epoch_id", "segment_index", "recv_time_ms", "recv_seq"]).groupby(group_cols, dropna=False):
        run_id = part["is_high_heavy"].ne(part["is_high_heavy"].shift(fill_value=False)).cumsum()
        target_part = part[part["is_high_heavy"]].copy()
        if target_part.empty:
            continue
        run_table = (
            target_part.assign(target_run_id=run_id[target_part.index])
            .groupby("target_run_id", observed=True)
            .agg(
                source_day=("source_day", "first"),
                epoch_id=("epoch_id", "first"),
                segment_index=("segment_index", "first"),
                start_ms=("recv_time_ms", "first"),
                end_ms=("recv_time_ms", "last"),
                start_ts=("ts", "first"),
                end_ts=("ts", "last"),
                observations=("is_high_heavy", "size"),
                duration_ms=("dt_ms", "sum"),
                entry_Z_label=("Z_label", "first"),
                start_mid=("mid", "first"),
                end_mid=("mid", "last"),
                mean_rho=("rho", "mean"),
                mean_lambda_rate=(f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms", "mean"),
            )
            .reset_index(drop=True)
        )
        rows.append(run_table)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


target_episodes = build_target_episodes(features_with_entries)
if not target_episodes.empty:
    target_episodes["delta_mid_episode"] = target_episodes["end_mid"] - target_episodes["start_mid"]

display(target_episodes.head())
display(
    target_episodes.groupby("entry_Z_label", observed=True)
    .agg(
        episodes=("duration_ms", "size"),
        mean_duration_ms=("duration_ms", "mean"),
        median_duration_ms=("duration_ms", "median"),
        p90_duration_ms=("duration_ms", lambda x: x.quantile(0.90)),
        mean_delta_mid_episode=("delta_mid_episode", "mean"),
    )
    .reset_index()
    .sort_values("episodes", ascending=False)
    if not target_episodes.empty else pd.DataFrame()
)


## 8. Entry Event Study

This aligns observations around entries into `high|heavy` and summarizes the average path before and after entry.

The output answers questions like:

- does activity rise before imbalance becomes heavy?
- does imbalance become heavy before activity reaches the high bucket?
- what happens to midprice around entry?


In [ ]:
def build_entry_event_study(frame: pd.DataFrame, entries: pd.DataFrame, offsets_ms: list[int], *, max_entries: int | None = None) -> pd.DataFrame:
    if entries.empty:
        return pd.DataFrame()

    entry_sample = entries.copy()
    if max_entries is not None and len(entry_sample) > max_entries:
        entry_sample = entry_sample.sample(max_entries, random_state=0).sort_values(["source_day", "epoch_id", "segment_index", "recv_time_ms"])

    rows = []
    group_cols = ["source_day", "epoch_id", "segment_index"]
    lambda_col = f"lambda_total_rate_{LAMBDA_WINDOW_MS}ms"
    for keys, group_entries in entry_sample.groupby(group_cols, dropna=False):
        source_day, epoch_id, segment_index = keys
        part = frame[
            (frame["source_day"] == source_day)
            & (frame["epoch_id"] == epoch_id)
            & (frame["segment_index"] == segment_index)
        ].sort_values(["recv_time_ms", "recv_seq"])
        if part.empty:
            continue

        times = part["recv_time_ms"].to_numpy(dtype="float64")
        mids = part["mid"].to_numpy(dtype="float64")
        rhos = part["rho"].to_numpy(dtype="float64")
        lambdas = part[lambda_col].to_numpy(dtype="float64")
        spreads = part["spread"].to_numpy(dtype="float64")
        z_labels = part["Z_label"].astype("string").to_numpy(dtype=object)
        buckets = part["lambda_total_bucket"].astype("string").to_numpy(dtype=object)

        for entry_id, entry in group_entries.reset_index(drop=True).iterrows():
            entry_time = float(entry["recv_time_ms"])
            entry_pos = np.searchsorted(times, entry_time, side="left")
            if entry_pos >= len(times):
                continue
            entry_mid = mids[entry_pos]
            for offset_ms in offsets_ms:
                target_time = entry_time + offset_ms
                pos = np.searchsorted(times, target_time, side="left")
                if pos >= len(times):
                    continue
                rows.append(
                    {
                        "source_day": source_day,
                        "epoch_id": epoch_id,
                        "segment_index": segment_index,
                        "entry_local_id": entry_id,
                        "entry_direction": entry["entry_direction"],
                        "offset_ms": offset_ms,
                        "mid_delta_from_entry": mids[pos] - entry_mid,
                        "rho": rhos[pos],
                        "lambda_total_rate": lambdas[pos],
                        "spread": spreads[pos],
                        "Z_label": z_labels[pos],
                        "lambda_total_bucket": buckets[pos],
                    }
                )
    return pd.DataFrame(rows)


event_study = build_entry_event_study(features_with_entries, entries, EVENT_STUDY_OFFSETS_MS, max_entries=MAX_EVENT_STUDY_ENTRIES)
event_study_summary = (
    event_study.groupby(["entry_direction", "offset_ms"], observed=True)
    .agg(
        observations=("mid_delta_from_entry", "size"),
        mean_mid_delta=("mid_delta_from_entry", "mean"),
        median_mid_delta=("mid_delta_from_entry", "median"),
        mean_rho=("rho", "mean"),
        mean_lambda_total_rate=("lambda_total_rate", "mean"),
        mean_spread=("spread", "mean"),
    )
    .reset_index()
    .sort_values(["entry_direction", "offset_ms"])
    if not event_study.empty else pd.DataFrame()
)

display(event_study_summary)

if not event_study_summary.empty:
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, constrained_layout=True)
    for entry_direction, part in event_study_summary.groupby("entry_direction", observed=True):
        part = part.sort_values("offset_ms")
        axes[0].plot(part["offset_ms"], part["mean_mid_delta"], marker="o", label=str(entry_direction))
        axes[1].plot(part["offset_ms"], part["mean_rho"], marker="o", label=str(entry_direction))
        axes[2].plot(part["offset_ms"], part["mean_lambda_total_rate"], marker="o", label=str(entry_direction))
    axes[0].axvline(0, color="black", linewidth=0.8, alpha=0.5)
    axes[1].axvline(0, color="black", linewidth=0.8, alpha=0.5)
    axes[2].axvline(0, color="black", linewidth=0.8, alpha=0.5)
    axes[0].set_ylabel("mean mid delta")
    axes[1].set_ylabel("mean rho")
    axes[2].set_ylabel("mean lambda rate")
    axes[2].set_xlabel("offset from entry, ms")
    axes[0].set_title("Event study around entry into high activity + heavy imbalance")
    for ax in axes:
        ax.legend()
    plt.show()


## 9. Interpretation Notes

Use this section after running the notebook.

Things to check:

- Which `from_joint_state` most often precedes `high|heavy`?
- Are entries mostly `strong buy-heavy`, `strong sell-heavy`, or balanced across both?
- Do high/heavy episodes last long enough to matter, or are they mostly one-row flickers?
- In the event study, does `lambda_total_rate` rise before `rho` becomes extreme, or does `rho` become extreme first?
- Does the midprice drift differently around buy-heavy entries versus sell-heavy entries?
